In [14]:
!pip install ucimlrepo

In [15]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
mushroom = fetch_ucirepo(id=73)

# data (as pandas dataframes)
X = mushroom.data.features
y = mushroom.data.targets

# metadata
print(mushroom.metadata)

# variable information
print(mushroom.variables)


{'uci_id': 73, 'name': 'Mushroom', 'repository_url': 'https://archive.ics.uci.edu/dataset/73/mushroom', 'data_url': 'https://archive.ics.uci.edu/static/public/73/data.csv', 'abstract': 'From Audobon Society Field Guide; mushrooms described in terms of physical characteristics; classification: poisonous or edible', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 8124, 'num_features': 22, 'feature_types': ['Categorical'], 'demographics': [], 'target_col': ['poisonous'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1981, 'last_updated': 'Thu Aug 10 2023', 'dataset_doi': '10.24432/C5959T', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': "This data set includes descriptions of hypothetical samples corresponding to 23 species of gilled mushrooms in the Agaricus and Lepiota Family (pp. 500-525).  Each species is identified as definitely edible, definitely po

In [16]:
# ============================
# Daten Angucken
# ============================

# Die Daten kommen von dem Repository: https://archive.ics.uci.edu/dataset/73/mushroom. Wir laden die Daten in der variable mushroom.
# Wichtig ist es immer sich mit solchen Daten etwas vertraut zu machen. Folgende Punkte koennen sind Ansaetze dafuer sich die heruntergeladenen Daten einmal anzusehen.

# 1) Was genau fuer ein Datentyp ist mushroom?
# 2) Wie genau sieht mushroom aus? Wie koennen wir die Daten benutzen?
# 3) Wie koennen wir die Daten in einem neuronalen Modell benutzen, bzw. wie koennen wir kategoriale Daten in ein NN geben?

#Tipp: Indem man in jupyternotebooks eine Zelle ausfuehrt, wird immer der letzte Ausdruck dieser Zelle geprintet. Oft gibt das uebersichtlicher formatierte Ausgaben als mit print().
#Tipp: Zellen lassen sich mit ctl + enter leicht ausfuehren.
X


,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,x,s,n,t,p,f,c,n,k,e,...,s,w,w,p,w,o,p,k,s,u
1,x,s,y,t,a,f,c,b,k,e,...,s,w,w,p,w,o,p,n,n,g
2,b,s,w,t,l,f,c,b,n,e,...,s,w,w,p,w,o,p,n,n,m
3,x,y,w,t,p,f,c,n,n,e,...,s,w,w,p,w,o,p,k,s,u
4,x,s,g,f,n,f,w,b,k,t,...,s,w,w,p,w,o,e,n,a,g
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8119,k,s,n,f,n,a,c,b,y,e,...,s,o,o,p,o,o,p,b,c,l
8120,x,s,n,f,n,a,c,b,y,e,...,s,o,o,p,n,o,p,b,v,l
8121,f,s,n,f,n,a,c,b,n,e,...,s,o,o,p,o,o,p,b,c,l
8122,k,y,n,f,y,f,c,n,b,t,...,k,w,w,p,w,o,e,w,v,l


In [17]:
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
import numpy as np
import torch

In [18]:
# ====================
# Preprocessing
# Transformiere die kategoriale Daten in One-Hot-Encodings
# Ignorier hier ruhig die Einzelheiten. Hier passiert erst einmal sehr spezifische 'library magic', die auf angenehme Weise genau diese Aufgabe erfuellt.
# ====================

ohe = preprocessing.OneHotEncoder(handle_unknown='ignore')

ohe.fit(X)
X_ohe = ohe.fit_transform(X).toarray()
ohe.fit(y)
y_ohe = ohe.fit_transform(y).toarray()
X_data = torch.tensor(X_ohe, dtype=torch.float32)
y_data = torch.tensor(y_ohe, dtype=torch.float32)


X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.33)


In [19]:
# =========================
# Daten wieder Ansehen
# =========================

# Wie sehen die Daten nach dem Preprocessing aus? Was fuer ein typ haben sie? Was fuer eine Form haben sie?

In [20]:
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.nn.functional as F


In [21]:
# ==============
# Batching
# ==============


# Wir wollen die Daten nun in batches laden. Dazu wollen wir features und targets zuerst in einen Tensor zusammenfuegen, um diese spaeter parallel laden zu koennen.
# DataLoader kann einen einfachen Tensor nehmen. Angenommen wir initialisieren einen DataLoader(t) wobei t den shape [561, 1, 256, 256] hat (das koennten Bilddaten sein), dann wuerde
# next(iter(data)) einen tensor mit [64, 1, 256, 256] geben (angenommen dass die batchsize 64 ist). Das heisst die erste Dimension muessen die verschiedenen Samples sein, von denen der Dataloader dann batches gibt.

# 1) Wie koennen wir unsere train- und test-Daten mit torch.cat() zusammenfuehren, um einen Dataloader fuer features und targets zu bekommen?
# train_dataloader = DataLoader(batch_size=64, shuffle=True)
# test_dataloader = DataLoader(batch_size=64, shuffle=True)


In [22]:
from tqdm import tqdm
%pip install wandb -q
import wandb


In [23]:
# =============
# Das Modell
# =============

hidden_size = 50

class ShroomClassifier(nn.Module):
  def __init__(self):
    super(ShroomClassifier, self).__init__()
    # Implementiere ein einfaches multilayer Perzeptron mit einem hidden Layer von groesse 50.
    # Denk daran, was soll der input shape sein? Was soll der Output shape sein?
    self.layer1 = nn.Linear(117, 50)
    self.layer2 = nn.Linear(50, 2)


  def forward(self, x):
    # Implementiere den Forward pass.
    # Benutze die Relu Funktion, um nonlinearitaet nach dem hidden Layer zu bekommen.
    x = self.layer1(x)
    x = F.relu(x)
    x = self.layer2(x)
    return x


In [24]:
# ==================
# Hyperparameter
# ==================

# Passe die Parameter eventuell an.

learning_rate = 1e-4
epochs = 100
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [25]:
# ==============
# Training und Testing
# ==============

def train_step(model, loader, loss, optimizer, lr=learning_rate):
  model.train()
  for batch in loader:
    optimizer.zero_grad()
    X_batch, y_batch = batch[:, :-2], batch[:, -2:] # Schoener waere hier evtl y_train.shape[1]. Wir trennen features und targets.
    X_batch, y_batch = X_batch.to(device), y_batch.to(device) # Tensoren koennen auf der cpu oder gpu geladen werden. Hier ist es wichtig immer die Daten auf demselben device wie das Modell zu haben!
    output = model(X_batch) # Forward pass

    batch_loss = loss(output, y_batch) # Ausrechnen des loss.
    batch_loss.backward() # Backpropagation, um die Parameter des Modells folgend anzupassen.
    optimizer.step() # Gradient descent step. Alle Tensoren und gradients existieren irgendwie im Hintergund und werden ueber pytorch gehandhabt.


def test_step(model, loader, loss):
  model.eval()
  test_loss = 0
  accuracy = 0
  with torch.no_grad():
    for batch in loader:
      X_batch, y_batch = batch[:, :-2], batch[:, -2:]
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)

      output = model(X_batch)
      test_loss += loss(output, y_batch)  # akkumulierter loss
      accuracy += torch.sum(torch.argmax(output, dim=1) == torch.argmax(y_batch, dim=1)) #akkumulierte accuracy

  test_loss = test_loss / len(loader)
  accuracy = accuracy / y_test.shape[0]
  wandb.log({"test/loss": test_loss, "test/accuracy": accuracy})
  print(f"Test loss: {test_loss:.4f}, Accuracy: {accuracy:.4f}")


In [26]:
# ====================
# Putting it together
# ====================


model = ShroomClassifier().to(device)
loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


wandb.login()
config = {"epochs": epochs,
          "learning rate": learning_rate,
          "hidden size": hidden_size,
          "activation function": "relu",
          "optimizer": "Adam"
          # ......
          }
wandb.init(project="Uebung pytorch", name="Mushroom Classifier", config=config)
for _ in tqdm(range(epochs)):
  train_step(model, train_dataloader, loss, optimizer)
  test_step(model, test_dataloader, loss)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


  0%|          | 0/100 [00:00<?, ?it/s]


NameError: name 'train_dataloader' is not defined